# garimpo-uncertainty on Colab

The README's install and quick start cells, plus the GPU test run of `docs/testing.md`. Run the
session cell at the start of every session, then one section. Never use Run all: each section
ends by releasing the runtime. Save a read-only GitHub token as the Colab secret `GH_TOKEN`
first. Why the cells are written this way: `docs/colab.md`.

| Section | Runtime | When |
|---|---|---|
| Tests | GPU | Once, before the first paid run (`docs/testing.md`) |
| Data | CPU | Once; writes `data.tar.zst` to Drive |
| Pilot | L4 | After the data; see `docs/pilot.md` |

## Session

Until `data.tar.zst` exists in Drive, the `tar` line prints an error and the cell carries on.

In [ ]:
import os
from google.colab import drive, userdata
drive.mount("/content/drive")
REPO = "github.com/meiazero/garimpo-uncertainty.git"         # the repository's path on GitHub
DRIVE = "/content/drive/MyDrive/garimpo"                    # store and runs between sessions
DATA = "/content/garimpo-data"                              # always this path: runs record it
TOKEN = userdata.get("GH_TOKEN")
os.environ["GARIMPO_DATA_DIR"] = DATA
os.environ["PATH"] = "/root/.local/bin:" + os.environ["PATH"]
!apt-get -qq install -y rsync zstd > /dev/null
!curl -LsSf https://astral.sh/uv/install.sh | sh
!git clone --recurse-submodules https://{TOKEN}@{REPO} /content/garimpo-uncertainty
%cd /content/garimpo-uncertainty
!uv sync
!mkdir -p {DRIVE}/runs {DATA} runs outputs/logs
!tar -I zstd -xf {DRIVE}/data.tar.zst -C {DATA}
!rsync -a {DRIVE}/runs/ runs/

## Tests

The E2E suite on synthetic data; it needs no download. Artifacts go to `tests/artifacts/`.

In [ ]:
!GARIMPO_TEST_DEVICE=cuda uv run --no-sync pytest -q

Release the GPU once you have read the failures, if any.

In [ ]:
from google.colab import runtime; runtime.unassign()

## Data

The last line releases the runtime.

In [ ]:
!date; uv run --no-sync python scripts/download_data.py metadata; date
!date; uv run --no-sync python scripts/download_data.py scenes --tiles outputs/tiles/tiles_1pct.json --resampling-20m bilinear; date
!date; uv run --no-sync python scripts/download_data.py scenes --tiles outputs/tiles/tiles_100pct.json --resampling-20m bilinear; date
!uv run --no-sync python scripts/download_data.py weights --encoder-weights imagenet sentinel2_moco
!tar --format=posix -I 'zstd -3 -T0' -cf {DRIVE}/data.tar.zst -C {DATA} .
from google.colab import runtime; runtime.unassign()

## Pilot

Read the verdict lines the cell prints, not the cell's status (`docs/colab.md`).

In [ ]:
!for w in none imagenet sentinel2_moco; do ENCODER_WEIGHTS=$$w bash slurm/pilot.sbatch 2>&1 | tee outputs/logs/pilot-$$w.log; done
!rsync -a runs/ {DRIVE}/runs/ && rsync -a outputs/logs/ {DRIVE}/logs/
from pathlib import Path
VERDICTS = ("PIPELINE CHECK: PASS", "PIPELINE CHECK: FAIL")
# The cell succeeds whatever the jobs did; their verdicts say how they went. Nothing here may
# raise, or the GPU would stay assigned.
for log in [Path(f"outputs/logs/pilot-{w}.log") for w in ("none", "imagenet", "sentinel2_moco")]:
    text = log.read_text(errors="replace") if log.exists() else ""
    verdict = [line for line in text.splitlines() if line.startswith(VERDICTS)]
    print(log.name, verdict[-1] if verdict else "NO VERDICT: the job or its check died; see the log")
from google.colab import runtime; runtime.unassign()